In [1]:
import pandas as pd
import re

INPUT_FILE = "train.xlsx"
OUTPUT_FILE = "train_br_ready_strict.xlsx"

TEXT_COL = "text_ar"
GLOSS_COL = "gloss_ar"

df = pd.read_excel(INPUT_FILE)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)

Shape: (102, 7)
Columns: ['id', 'text_ar', 'gloss_ar', 'source_type', 'aug_method', 'parent_id', 'is_augmented']


,id,text_ar,gloss_ar,source_type,aug_method,parent_id,is_augmented
0,1,انتبه,انتبه خطر تمام,original,none,1,0
1,2,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,original,none,2,0
2,3,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,original,none,3,0
3,4,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,original,none,4,0
4,6,6 شهر,6 شهر,original,none,6,0
5,7,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,original,none,7,0
6,8,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,original,none,8,0
7,9,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,original,none,9,0
8,10,بعد الاكل,اكل خلص فورا دواء تمام,original,none,10,0
9,13,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,original,none,13,0


In [2]:
required_cols = ["id", TEXT_COL, GLOSS_COL]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("All required columns are present.")

All required columns are present.


In [12]:
def normalize_ar(text: str) -> str:
    text = str(text).strip()

    # إزالة التشكيل
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)

    # توحيد الحروف
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)

    # توحيد المسافات
    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(r"\bو(الاكل|اكل|الطعام|طعام|وجبة|وجبات|الفطور|فطور|الغدا|غدا|العشا|عشا)\b", r"و \1", text)

    return text


def tokenize(text: str):
    return normalize_ar(text).split()


def normalize_set(words_set):
    return {normalize_ar(w) for w in words_set}

In [13]:
RAW_SAFE_REPLACEABLE_WORDS = {
    "اكل", "الاكل",
    "طعام", "الطعام",
    "وجبة", "وجبات",
    "فطور", "الفطور",
    "غدا", "الغدا",
    "عشا", "العشا"
}

SAFE_REPLACEABLE_WORDS = normalize_set(RAW_SAFE_REPLACEABLE_WORDS)
SAFE_REPLACEABLE_WORDS

{'اكل',
 'الاكل',
 'الطعام',
 'العشا',
 'الغدا',
 'الفطور',
 'طعام',
 'عشا',
 'غدا',
 'فطور',
 'وجبات',
 'وجبة'}

In [14]:
RAW_PROTECTED_WORDS = {
    # نفي / نهي / تحذير
    "لا", "ليس", "مو", "بدون", "ممنوع", "ولا", "انتبه", "احذر",

    # شرط وربط
    "اذا", "واذا", "إذا", "وإذا", "لو", "لما", "عند", "عندما",

    # اتجاه / زمن / علاقة
    "قبل", "بعد", "مع", "ضد", "ثم", "ورا", "خلال", "اثناء", "أثناء",

    # التكرار والعدد
    "مرة", "مره", "مرتين", "مرتان", "مرات",
    "ثلاث", "ثلاثة", "ثلاثه", "تلات",
    "اربع", "أربع", "خمس", "خمسة", "ست", "سبع", "ثمان", "تسع", "عشر",

    # الوقت والمدة
    "يوم", "يوميا", "يوميًا", "ايام", "أيام",
    "اسبوع", "أسبوع", "أسبوعين", "اسبوعين",
    "شهر", "شهور", "اشهر",
    "مدة", "مده", "لمدة", "لمده", "لمدّة",
    "ساعة", "ساعه", "ساعات",
    "دقيقة", "دقيقه", "دقائق", "دقايق",
    "صباحا", "صباحًا", "مساء", "ليلا", "ليلًا", "الصبح", "المسا",
    "اليوم", "الليلة", "الليله", "باليوم", "بالاسبوع",

    # الجرعة والكمية
    "جرعة", "الجرعة", "جرعات",
    "حبة", "حبّة", "حبه", "الحبة", "الحبه", "حبتين",
    "كبسولة", "كبسوله", "كبسولتين",
    "قرص", "أقراص", "اقراص",
    "نقطة", "نقطه", "نقطتين", "قطرة", "قطره", "قطرتين",
    "مل", "ملل", "مليلتر", "ملغ", "جرام", "غرام",
    "ملعقة", "ملعقه", "معلقة", "معلقه",
    "نصف", "نص", "ربع", "كل", "بنص", "بساعة", "بكل",

    # طريقة الاستخدام
    "الفم", "فمويا", "فمويًا", "فموي", "عن", "طريق",
    "حقن", "وريد", "وريدي", "عضل", "عضلي",
    "استنشاق", "استنشق", "بخ", "بخة", "بختين",
    "موضعي", "موضعيا", "موضعيًا",
    "بلع", "ابتلع", "يمضغ", "مضغ", "يذوب",

    # حمل / حساسية / نزيف
    "حمل", "بالحمل", "الحمل", "مرضع", "رضاعة", "الرضاعة",
    "حساسية", "حساسيه", "تحسس",
    "نزيف", "ينزف", "نزف",

    # أمراض / أعراض / حالات
    "الالم", "ألم", "الم", "الألم",
    "الحرارة", "الحراره", "حرارة", "حراره",
    "السكر", "سكر", "الضغط", "ضغط",
    "الاعراض", "اعراض", "اعراضك", "العرض", "عرض",
    "الالتهاب", "التهاب",
    "حمى", "سعال", "حكة", "حكه", "احمرار", "تورم",
    "دوخة", "دوخه", "اسهال", "إسهال", "امساك", "إمساك",
    "قيء", "استفراغ", "غثيان", "صداع", "دوار",

    # أعضاء أو مواضع
    "الرأس", "الراس", "عين", "العين", "الاذن", "الأذن",
    "الانف", "الأنف", "الفم", "الحلق", "الصدر", "البطن", "المعدة", "المعده",
    "الجلد", "يد", "اليد", "رجل", "الرجل", "الظهر",

    # كلمات عامة نريد منعها من الاستبدال
    "ممكن", "يعملك", "شوية", "تقلق", "ضروري", "خطر",
    "استخدام", "استخدامه", "استخدم",
    "اشرب", "اشربه", "استمر", "استمرت", "تجنب", "توقفه",
    "الاطفال", "الطفل", "الطبيب", "الدكتور",
    "الحبوب", "البخاخ", "العبوة", "العلبة", "المكيال",
    "عالريق", "عالمعدة", "بالاكل", "بالمسا",
    "هاد", "هذا", "هاي", "هذي", "هي", "اول", "أول", "لانو", "لانه",
    "مشان", "عشان", "عليك", "عليه", "علي", "بسبب", "تعود", "بينعس",

    # ألفاظ دوائية عامة
    "دواء", "الدواء", "دوا", "الدوا",
    "علاج", "العلاج",
    "مرهم", "المرهم",
    "كريم", "الكريم",
    "شراب", "الشراب"
}

PROTECTED_WORDS = normalize_set(RAW_PROTECTED_WORDS)
len(PROTECTED_WORDS)

241

In [15]:
replaceable_positions = []
replaceable_tokens = []
protected_tokens_found = []
br_ready = []

for tokens in df[TEXT_COL].apply(tokenize):
    row_positions = []
    row_replaceable = []
    row_protected = []

    for i, tok in enumerate(tokens):
        tok_norm = normalize_ar(tok)

        if tok_norm in PROTECTED_WORDS:
            row_protected.append(tok_norm)

        # لا نسمح بالاستبدال إلا إذا كانت الكلمة ضمن القائمة الآمنة فقط
        if tok_norm in SAFE_REPLACEABLE_WORDS:
            row_positions.append(i)
            row_replaceable.append(tok_norm)

    replaceable_positions.append(row_positions)
    replaceable_tokens.append(row_replaceable)
    protected_tokens_found.append(sorted(set(row_protected)))
    br_ready.append(1 if len(row_positions) > 0 else 0)

In [16]:
df["text_norm"] = df[TEXT_COL].apply(normalize_ar)
df["gloss_norm"] = df[GLOSS_COL].apply(normalize_ar)

df["tokens_text"] = df["text_norm"].apply(tokenize)
df["tokens_gloss"] = df["gloss_norm"].apply(tokenize)

df["replaceable_positions"] = replaceable_positions
df["replaceable_tokens"] = replaceable_tokens
df["protected_tokens_found"] = protected_tokens_found
df["br_ready"] = br_ready

df.head(10)

,id,text_ar,gloss_ar,source_type,aug_method,parent_id,is_augmented,text_norm,gloss_norm,tokens_text,tokens_gloss,replaceable_positions,replaceable_tokens,protected_tokens_found,br_ready
0,1,انتبه,انتبه خطر تمام,original,none,1,0,انتبه,انتبه خطر تمام,[انتبه],"[انتبه, خطر, تمام]",[],[],[انتبه],0
1,2,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,original,none,2,0,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,"[حبة, 3, مرة, قبل, الاكل, بنص, ساعة, .]","[3, مرة, اكل, قبل, نص, ساعة, حبة]",[4],[الاكل],"[بنص, حبة, ساعة, قبل, مرة]",1
2,3,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,original,none,3,0,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,"[حبة, 3, مرة, قبل, الاكل, .]","[3, مرة, اكل, قبل, حبة]",[4],[الاكل],"[حبة, قبل, مرة]",1
3,4,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,original,none,4,0,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,"[حبة, 3, مرة, قبل, الاكل, بساعة, .]","[3, مرة, اكل, قبل, ساعة, حبة]",[4],[الاكل],"[بساعة, حبة, قبل, مرة]",1
4,6,6 شهر,6 شهر,original,none,6,0,6 شهر,6 شهر,"[6, شهر]","[6, شهر]",[],[],[شهر],0
5,7,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,original,none,7,0,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,"[نقطتين, بكل, طرف]","[اذن, يمين, نقطة, -, نقطة, اذن, يسار, نقطة, -,...",[],[],"[بكل, نقطتين]",0
6,8,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,original,none,8,0,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,"[لمدة, ثلاثة, يوم]","[استمرار, ثلاثة, يوم, بس, تمام]",[],[],"[ثلاثة, لمدة, يوم]",0
7,9,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,original,none,9,0,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,"[خده, ورا, الاكل]","[اكل, بعد, حبة, تمام, معدة, فاضية, لا]",[2],[الاكل],[ورا],1
8,10,بعد الاكل,اكل خلص فورا دواء تمام,original,none,10,0,بعد الاكل,اكل خلص فورا دواء تمام,"[بعد, الاكل]","[اكل, خلص, فورا, دواء, تمام]",[1],[الاكل],[بعد],1
9,13,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,original,none,13,0,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,"[ممكن, يعملك, شوية, اسهال, ,, لا, تقلق, .]","[انتبه, تمام, جسم, تعب, تمام, اسهال, امساك, دو...",[],[],"[اسهال, تقلق, شوية, لا, ممكن, يعملك]",0


In [17]:
total_rows = len(df)
ready_rows = int(df["br_ready"].sum())
not_ready_rows = total_rows - ready_rows

print("Total rows:", total_rows)
print("Rows ready for strict BR:", ready_rows)
print("Rows NOT ready:", not_ready_rows)

Total rows: 102
Rows ready for strict BR: 14
Rows NOT ready: 88


In [18]:
preview_cols = [
    "id",
    TEXT_COL,
    GLOSS_COL,
    "replaceable_tokens",
    "protected_tokens_found",
    "br_ready"
]

df[df["br_ready"] == 1][preview_cols].head(30)

,id,text_ar,gloss_ar,replaceable_tokens,protected_tokens_found,br_ready
1,2,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,[الاكل],"[بنص, حبة, ساعة, قبل, مرة]",1
2,3,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,[الاكل],"[حبة, قبل, مرة]",1
3,4,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,[الاكل],"[بساعة, حبة, قبل, مرة]",1
7,9,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,[الاكل],[ورا],1
8,10,بعد الاكل,اكل خلص فورا دواء تمام,[الاكل],[بعد],1
44,58,"قبل الفطور عالريق , خود حبة كل يوم .",صباح بكير فطور قبل حبة معدة فاضية,[الفطور],"[حبة, عالريق, قبل, كل, يوم]",1
48,62,"بعد وجبة دسمة عالصبحية , متل البيض والحليب وزي...",فطور دسم ثقيل اكل لازم تمام بيض حليب زيت - زيت...,[وجبة],[بعد],1
49,63,حبة قبل الفطور والغدا والعشا بنص ساعة .,فطور غدا عشا قبل حبة,"[الفطور, الغدا, العشا]","[بنص, حبة, ساعة, قبل]",1
55,71,خود حبة يوميا بعد الاكل,كل يوم حبة واحد اكل بعد تمام,[الاكل],"[بعد, حبة, يوميا]",1
56,72,حبة الصبح بعد الفطور,كل يوم صباح اكل بعد حبة,[الفطور],"[الصبح, بعد, حبة]",1


In [19]:
df.to_excel(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Final shape:", df.shape)

Saved: train_br_ready_strict.xlsx
Final shape: (102, 15)
